In [1]:
!pip install pandas


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np


# Inspecting Original Data set

In [3]:
#Loading data according to folder hirearchy 
df_original = pd.read_csv('../Data/raw/melbourne_ebike_trips_2024.csv')

In [4]:
#inspecting the data
df_original.head()

,trip_id,start_time,end_time,start_station,end_station,user_type,distance_km,bike_type,temperature_c,rain_mm,trip_duration_min
0,1,2024-10-29 22:00:00,2024-10-29 22:10:24,Richmond,Carlton,Subscriber,2.14,Standard,13.8,0.6,10.4
1,2,2024-02-05 20:00:00,2024-02-05 20:02:30,Richmond,Southbank,Subscriber,0.87,Electric,15.0,3.7,2.5
2,3,2024-08-12 14:00:00,2024-08-12 14:06:54,St Kilda,Carlton,Subscriber,2.50,Electric,21.9,2.8,6.9
3,4,2024-08-04 07:00:00,2024-08-04 07:07:12,Docklands,CBD Central,Subscriber,2.70,Electric,16.4,4.4,7.2
4,5,2024-08-26 22:00:00,2024-08-26 22:02:48,Richmond,Southbank,Subscriber,1.09,Electric,21.4,4.8,2.8


In [5]:
#Exploring the data 

###Check List
#shape
#dtypes
#missing values
#duplicates
#validation 
###

In [6]:
df_original.info()
#11 columns,5k rows , 5str,4 float64 ,1 int64

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   trip_id            5000 non-null   int64  
 1   start_time         5000 non-null   str    
 2   end_time           5000 non-null   str    
 3   start_station      5000 non-null   str    
 4   end_station        5000 non-null   str    
 5   user_type          5000 non-null   str    
 6   distance_km        5000 non-null   float64
 7   bike_type          5000 non-null   str    
 8   temperature_c      5000 non-null   float64
 9   rain_mm            5000 non-null   float64
 10  trip_duration_min  5000 non-null   float64
dtypes: float64(4), int64(1), str(6)
memory usage: 429.8 KB


In [7]:

df_original.duplicated().sum()

np.int64(0)

In [8]:
print(df_original.isna().sum())


columns_name_list = df_original.columns.to_list()
df_original[columns_name_list].nunique()

trip_id              0
start_time           0
end_time             0
start_station        0
end_station          0
user_type            0
distance_km          0
bike_type            0
temperature_c        0
rain_mm              0
trip_duration_min    0
dtype: int64


trip_id              5000
start_time           3788
end_time             4995
start_station           7
end_station             7
user_type               2
distance_km          1068
bike_type               2
temperature_c         336
rain_mm                82
trip_duration_min     521
dtype: int64

In [9]:
# min and maxes to make sure everything makes sense for every value

print(df_original[columns_name_list].min())
df_original[columns_name_list].max()


trip_id                                1
start_time           2024-01-01 01:00:00
end_time             2024-01-01 01:09:24
start_station                CBD Central
end_station                  CBD Central
user_type                         Casual
distance_km                         0.02
bike_type                       Electric
temperature_c                       -8.8
rain_mm                              0.0
trip_duration_min                    0.1
dtype: object


trip_id                             5000
start_time           2024-12-30 22:00:00
end_time             2024-12-30 22:01:42
start_station                   St Kilda
end_station                     St Kilda
user_type                     Subscriber
distance_km                        20.44
bike_type                       Standard
temperature_c                       39.3
rain_mm                              9.7
trip_duration_min                   94.4
dtype: object

In [10]:
#Initial Observation Report

###Normal
#Data set is farly clean
#values make sense
#no duplicated or null values 
#data types are correct and will allow for manipulation

###Further investigation/cleaning
#start time and end time will need to be transformed according to needs (change dtype to datetime, seperate values into columns for easier use )
#min temprature is recorde at -8.8 which seems slighty strange but not impossible , will investigate during cleaning
# min trip duration time is 10 secodns while min distance is 20 meters , this seems fine but will investiagte just in case 

# Data Cleaning

In [11]:
#now that we are moving on to the cleaning phase ,i will create a copy of the raw dataset ,so that we dont damage or temper with the original
#before ....

df_copy = df_original.copy()

In [12]:
#First we will change the data type of start time and end time , so that when we seperate them they will be in the right format , this will save us
#extra formating later 

df_copy["start_time"]= pd.to_datetime(df_copy["start_time"],format="%Y-%m-%d %H:%M:%S")
df_copy["end_time"]= pd.to_datetime(df_copy["end_time"],format="%Y-%m-%d %H:%M:%S")

df_copy.dtypes

trip_id                       int64
start_time           datetime64[us]
end_time             datetime64[us]
start_station                   str
end_station                     str
user_type                       str
distance_km                 float64
bike_type                       str
temperature_c               float64
rain_mm                     float64
trip_duration_min           float64
dtype: object

In [13]:
#Seperating start time into date and time, used variable names s_time and s_date in order not to override the orignal start_time column
#we might drop the column later and rename the time and date columns but for now we keep it as this 

#df_copy["s_date"] = df_copy["start_time"].dt.date
#df_copy["s_time"] = df_copy["start_time"].dt.time

#df_copy.head()


### learnign point : after havign seperated the values in start time and endtime into date and time , i have come to realise that their dtype changes
#from datetime to object and after having a further look ive come to reaslise that its best to jsut keep it as date time not only for speed 
# but also to avoid redundancy as long as start time and date time are of the datetime dtype we can access any value inside indivudually regardless 

In [14]:
#Having come to the previous realisation , ive come to my next few steps:

#create a new column that determiens wether its weekday or a weekend (this will be usefull for analysis later)
#validate/ handle negative trip durations
#validate wether a trips end time is always greater than its start time 

In [15]:
#the reason behind chosing start time rather then end time , is because for analysis we will most likely want to know when the most rides happen 
#aka start time , but if need later can use end time

#also i decided rather then just 1 that tells me wether its weekend or weekday, i will also create a column that tells me what each day was 
#so that if we want to do any deeper grouping lets say rides on "monday" we are able to do so 

df_copy["day"] = df_copy["start_time"].dt.day_name()
df_copy["day_type"] = np.where(df_copy["start_time"].dt.weekday > 4, 'Weekend', 'Weekday')
df_copy.head()


,trip_id,start_time,end_time,start_station,end_station,user_type,distance_km,bike_type,temperature_c,rain_mm,trip_duration_min,day,day_type
0,1,2024-10-29 22:00:00,2024-10-29 22:10:24,Richmond,Carlton,Subscriber,2.14,Standard,13.8,0.6,10.4,Tuesday,Weekday
1,2,2024-02-05 20:00:00,2024-02-05 20:02:30,Richmond,Southbank,Subscriber,0.87,Electric,15.0,3.7,2.5,Monday,Weekday
2,3,2024-08-12 14:00:00,2024-08-12 14:06:54,St Kilda,Carlton,Subscriber,2.50,Electric,21.9,2.8,6.9,Monday,Weekday
3,4,2024-08-04 07:00:00,2024-08-04 07:07:12,Docklands,CBD Central,Subscriber,2.70,Electric,16.4,4.4,7.2,Sunday,Weekend
4,5,2024-08-26 22:00:00,2024-08-26 22:02:48,Richmond,Southbank,Subscriber,1.09,Electric,21.4,4.8,2.8,Monday,Weekday


In [16]:
#creating a month column

df_copy["month"] = df_copy["start_time"].dt.month

In [17]:
df_copy.head()

,trip_id,start_time,end_time,start_station,end_station,user_type,distance_km,bike_type,temperature_c,rain_mm,trip_duration_min,day,day_type,month
0,1,2024-10-29 22:00:00,2024-10-29 22:10:24,Richmond,Carlton,Subscriber,2.14,Standard,13.8,0.6,10.4,Tuesday,Weekday,10
1,2,2024-02-05 20:00:00,2024-02-05 20:02:30,Richmond,Southbank,Subscriber,0.87,Electric,15.0,3.7,2.5,Monday,Weekday,2
2,3,2024-08-12 14:00:00,2024-08-12 14:06:54,St Kilda,Carlton,Subscriber,2.50,Electric,21.9,2.8,6.9,Monday,Weekday,8
3,4,2024-08-04 07:00:00,2024-08-04 07:07:12,Docklands,CBD Central,Subscriber,2.70,Electric,16.4,4.4,7.2,Sunday,Weekend,8
4,5,2024-08-26 22:00:00,2024-08-26 22:02:48,Richmond,Southbank,Subscriber,1.09,Electric,21.4,4.8,2.8,Monday,Weekday,8


In [18]:
# Here im unsure of how to go about it i can validate impossible distances in 2 ways , 1 by checking the min and max which i have 
#and making sure they make sense compared ot the ditance , i mean obisouly you dont actually have to be moving the whole time so thats something to
#keep in mind , but another thing i can do is the opposite and ehck the min and see if the distanced covered in that short period of time seems 
#impossible problem is i know i can use a double condition "and" to locate columns for example find me columns that  trip duration is less than such
# and trip distance greater than such , but im just not sure waht values to put in which 
#aside form that i can just do min max and see if they seem to impossible which for this data set those values seemed fine 
df_copy["trip_duration_min"].min()

valus_list_trip_dur = df_copy.loc[df_copy['trip_duration_min'] <= 1]



In [19]:
# validating wehter a trips end time is always greater tahn its start time 

#my arpoach is simply getting the difference and then pritning any negative values
# i know that we could just check trip duration because thats already the difference but for pratice sake ill do this 
#hence i wont be including this column in the final csv export just in this notebook as analysis 
trip_dif = df_copy["end_time"] - df_copy["start_time"]
df_copy[trip_dif < pd.Timedelta(0)]




,trip_id,start_time,end_time,start_station,end_station,user_type,distance_km,bike_type,temperature_c,rain_mm,trip_duration_min,day,day_type,month


In [20]:
#Data cleaning Report
#over all a fairly clean data set , minimal changes were needed , now that we have seperated date time into more usable columns and adjusted
#its data type we can move on to analysis 
#no null values 
#all new colunmns needed for analysis have been created 
#no impossible values 

In [ ]:
#Exporting
#df_copy.to_csv(r" data_cleaned.csv", index=False)
